# Chapter 7 - Lab 2: <font color='blue'>Leader-Follower with the OpenAI Agents SDK</font>

**<font color='purple'>Goal</font>**:
Solve the same financial-analysis task as Lab 1, but with a different architectural style: a **manager agent** dynamically routes requests to **three specialist worker agents** via *handoffs*. Routing decisions are made by the LLM at runtime, not declared statically.

This contrasts directly with the sequential pipeline in Lab 1. The same task, a different shape of coordination.

**<font color='purple'>Tech stack</font>**:

* **OpenAI Agents SDK** (`openai-agents`) — `Agent`, `Runner`, `function_tool`, and the   `handoffs=[...]` parameter.
* **OpenAI** `gpt-4o`.
* **Stub data** — same canned data as Lab 1.

## 1. Install packages

In [ ]:
%pip install -q openai-agents pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume the env var is already set.
    pass

## 3. Bootstrap the shared models and helpers

Chapter 7 labs share a `common.py` module that defines the Pydantic models and helpers used across the chapter. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%207/common.py',
        'common.py',
    )

from common import stub_lookup

print('Stub data loaded.')

## 4. Define the three specialist tools

Each tool wraps a piece of canned data from `common.py`. In production these would call Financial Modeling Prep; the tool *signatures* are the same so swapping in the real implementation is a drop-in change.

In [ ]:
def get_stock_price(symbol: str) -> dict:
    """Current price, volume, 50-/200-day averages, EPS, P/E, next earnings."""
    return stub_lookup(symbol)['stock_price']


def get_company_financials(symbol: str) -> dict:
    """Industry, sector, company name, and market capitalization."""
    return stub_lookup(symbol)['company']


def get_income_statement(symbol: str) -> dict:
    """Last income statement: revenue, gross profit, net income, EBITDA, EPS."""
    return stub_lookup(symbol)['income']

## 5. Define the three specialist worker agents

Each worker has a tightly scoped instruction and one tool. They are the *followers* in the leader-follower architecture — only the manager will call them, and only when relevant.

In [ ]:
from agents import Agent, Runner, function_tool

MODEL = 'gpt-4o'

stock_price_tool         = function_tool(get_stock_price)
company_financials_tool  = function_tool(get_company_financials)
income_statement_tool    = function_tool(get_income_statement)

stock_price_agent = Agent(
    name='Stock Price Agent',
    instructions=(
        'Fetch historical prices and trading metrics for a symbol: '
        'current price, volume, 50d/200d averages, EPS, P/E, next earnings.'
    ),
    tools=[stock_price_tool],
    model=MODEL,
)

company_basic_information_agent = Agent(
    name='Company Basic Information Agent',
    instructions='Fetch industry, sector, company name, market cap for a symbol.',
    tools=[company_financials_tool],
    model=MODEL,
)

income_statement_agent = Agent(
    name='Income Statement Agent',
    instructions='Fetch the latest income statement for a symbol.',
    tools=[income_statement_tool],
    model=MODEL,
)

## 6. Define the manager agent (leader)

The manager has no tools of its own — its job is **routing**. The `handoffs` parameter lists the worker agents it can delegate to. The LLM decides at runtime which worker matches the user's request.

In [ ]:
manager_agent = Agent(
    name='Manager Agent',
    instructions=(
        'Determine which agent is best suited to handle the user request, '
        'and transfer the conversation to that agent.'
    ),
    handoffs=[
        stock_price_agent,
        company_basic_information_agent,
        income_statement_agent,
    ],
    model=MODEL,
)

## 7. Run the agent and inspect the routing decision

Send a query that requires data from one of the specialists. Watch the manager hand off — the response will include both the routing decision and the final answer.

In [ ]:
query = 'What is the latest income statement for AAPL?'
result = await Runner.run(manager_agent, query)
print(result.final_output)

## 8. Results

Same task as Lab 1; very different architecture. **What to notice about leader-follower:**

* **Dynamic routing.** The LLM picks the right worker at runtime. Adding a new specialist is   one new `Agent` plus one entry in the manager's `handoffs` list.
* **No shared state.** Information flows through handoff messages and the manager's context.   Simpler than LlamaIndex's `Context`, but harder to audit a long history.
* **Trade-off.** Routing depends on prompt quality. If the manager's instructions are vague,   the wrong worker may be picked. The chapter discusses guardrails for production use.